# 01_07_agr_tariff_path_two_ids

Диагностическая тетрадка для трассировки полной цепочки:

`agr_id -> (agreements + active SA filter + AGR_TERMS P) -> r2_ip_merchants.id -> c_tariff_plan -> tariff_name -> commission_monthly_fix`

Фокус-кейсы:
- `413636181589`
- `508492576892`

Тетрадка read-only: ничего не создает и не изменяет в Озере.

In [ ]:
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Конфиг
report_month = '2026-04-01'  # можно менять
run_invalidate_metadata = True

agr_id_list = [
    '413636181589',
    '508492576892',
]

a_ids = [str(x).strip() for x in agr_id_list if str(x).strip()]
if not a_ids:
    raise RuntimeError('agr_id_list пустой')

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')

print('report_month =', report_month_label)
print('month_start =', month_start)
print('month_end =', month_end)
print('agr_id_list =', a_ids)

In [ ]:
# Подключение к Impala
if 'imp' in globals() and imp is not None:
    print('Using existing imp connection from current session')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'}
    )

try:
    imp._init_connection()
except Exception:
    pass

print('Impala connection initialized')

invalidate_tables = [
    'ods_alpha.scd1_agreements',
    'ods_alpha.scd1_companies',
    'ods_alpha.scd1_agr_terms',
    'ods.scd1_z_r2_ip_merchants',
    'ods.scd1_z_r2_tariff_plan',
    'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix',
]

if run_invalidate_metadata:
    with imp:
        for t in invalidate_tables:
            try:
                imp.execute(f'invalidate metadata {t}')
                imp.execute(f'refresh {t}')
                print(f'[invalidate ok] {t}')
            except Exception as e:
                print(f'[invalidate fail] {t}: {type(e).__name__}')
else:
    print('Invalidate skipped')

In [ ]:
# Вспомогательные функции

def sql_in_str(values):
    vals = [str(v).strip() for v in values if str(v).strip()]
    return ', '.join([f"'{v}'" for v in vals]) if vals else "''"


def run_impala(sql_text, mem_limit='8g'):
    with imp:
        imp.execute(f'set MEM_LIMIT={mem_limit}')
        return imp.fetch(sql_text)


a_ids_sql = sql_in_str(a_ids)
print('a_ids_sql =', a_ids_sql)

In [ ]:
# 1) История по договорам в agreements
sql_agreements_history = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.acq_class as string) as acq_class,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  coalesce(cast(a.ods_deleted_flg as string), '0') as ods_deleted_flg
from ods_alpha.scd1_agreements a
where cast(a.abs_agr_id as string) in ({a_ids_sql})
order by agr_id, d_valid_from desc, n_agr desc
"""

agreements_history_df = run_impala(sql_agreements_history, mem_limit='8g')
if agreements_history_df is None:
    agreements_history_df = pd.DataFrame()

print('agreements_history rows =', len(agreements_history_df))
display(agreements_history_df)

In [ ]:
# 2) Активные SA-договоры в месяце + проверка AGR_TERMS(cf_ter_type='P')
sql_active_sa_with_terms = f"""
with agr_base as (
  select
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.c_agr_number as string) as contract_number,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to,
    upper(trim(cast(a.acq_class as string))) as acq_class,
    coalesce(cast(a.ods_deleted_flg as string), '0') as ods_deleted_flg
  from ods_alpha.scd1_agreements a
  where cast(a.abs_agr_id as string) in ({a_ids_sql})
),
terms_active as (
  select distinct cast(t.n_agr as string) as n_agr
  from ods_alpha.scd1_agr_terms t
  where cast(t.d_valid_from as date) <= cast('{month_end}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
    and upper(trim(cast(t.cf_ter_type as string))) = 'P'
    and coalesce(cast(t.ods_deleted_flg as string), '0') <> '1'
)
select
  b.*,
  case
    when b.acq_class = 'SA'
      and b.d_valid_from <= cast('{month_end}' as date)
      and (b.d_valid_to is null or b.d_valid_to >= cast('{month_start}' as date))
      and b.ods_deleted_flg <> '1'
      then 1 else 0
  end as is_sa_active_in_month,
  case when ta.n_agr is not null then 1 else 0 end as has_active_p_term
from agr_base b
left join terms_active ta
  on ta.n_agr = b.n_agr
order by b.agr_id, b.d_valid_from desc, b.n_agr desc
"""

active_sa_terms_df = run_impala(sql_active_sa_with_terms, mem_limit='8g')
if active_sa_terms_df is None:
    active_sa_terms_df = pd.DataFrame()

print('active_sa_terms rows =', len(active_sa_terms_df))
display(active_sa_terms_df)

In [ ]:
# 3) История в R2 merchants: agr_id -> c_tariff_plan
sql_r2_merchants_history = f"""
select
  cast(m.id as string) as agr_id,
  cast(m.c_cl_org as string) as cft_id,
  cast(m.c_depart as string) as c_depart,
  cast(m.c_tariff_plan as string) as c_tariff_plan,
  coalesce(cast(m.ods_deleted_flg as string), '0') as ods_deleted_flg,
  row_number() over (
    partition by cast(m.id as string)
    order by cast(m.c_tariff_plan as string) desc
  ) as rn_by_plan_desc
from ods.scd1_z_r2_ip_merchants m
where cast(m.id as string) in ({a_ids_sql})
order by agr_id, rn_by_plan_desc, c_tariff_plan
"""

r2_merchants_df = run_impala(sql_r2_merchants_history, mem_limit='8g')
if r2_merchants_df is None:
    r2_merchants_df = pd.DataFrame()

print('r2_merchants rows =', len(r2_merchants_df))
display(r2_merchants_df)

In [ ]:
# 4) Справочник тарифных планов: c_tariff_plan -> tariff_name
plan_scope = []
if 'r2_merchants_df' in globals() and r2_merchants_df is not None and len(r2_merchants_df):
    if 'c_tariff_plan' in r2_merchants_df.columns:
        plan_scope = sorted([
            str(x).strip() for x in r2_merchants_df['c_tariff_plan'].dropna().astype(str).tolist()
            if str(x).strip() and str(x).strip().lower() != 'nan'
        ])
        plan_scope = sorted(set(plan_scope))

if not plan_scope:
    tariff_plan_df = pd.DataFrame(columns=['c_tariff_plan', 'tariff_name'])
else:
    plan_sql = sql_in_str(plan_scope)
    sql_tariff_plan = f"""
    select
      cast(tp.id as string) as c_tariff_plan,
      cast(tp.c_name as string) as tariff_name
    from ods.scd1_z_r2_tariff_plan tp
    where cast(tp.id as string) in ({plan_sql})
    order by c_tariff_plan
    """
    tariff_plan_df = run_impala(sql_tariff_plan, mem_limit='8g')
    if tariff_plan_df is None:
        tariff_plan_df = pd.DataFrame(columns=['c_tariff_plan', 'tariff_name'])

print('plan_scope size =', len(plan_scope))
display(tariff_plan_df)

In [ ]:
# 5) Детализация тарифа: c_tariff_plan -> c_tariff -> c_summa (как в 08b)
if not plan_scope:
    tariff_tune_fix_details_df = pd.DataFrame(columns=['c_tariff_plan', 'c_tariff', 'tariff_fix_id', 'commission_monthly_fix', 'rn_sum_desc'])
    tariff_fix_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])
else:
    plan_sql = sql_in_str(plan_scope)
    sql_tune_fix_details = f"""
    select
      cast(tt.c_tariff_plan as string) as c_tariff_plan,
      cast(tt.c_tariff as string) as c_tariff,
      cast(tf.id as string) as tariff_fix_id,
      cast(tf.c_summa as decimal(18,2)) as commission_monthly_fix,
      row_number() over (
        partition by cast(tt.c_tariff_plan as string)
        order by cast(tf.c_summa as decimal(18,2)) desc, cast(tf.id as string) desc
      ) as rn_sum_desc
    from ods.scd1_z_r2_tariff_tune tt
    left join ods.scd1_z_r2_tariff_fix tf
      on tt.c_tariff = tf.id
    where cast(tt.c_tariff_plan as string) in ({plan_sql})
    order by c_tariff_plan, rn_sum_desc
    """
    tariff_tune_fix_details_df = run_impala(sql_tune_fix_details, mem_limit='8g')
    if tariff_tune_fix_details_df is None:
        tariff_tune_fix_details_df = pd.DataFrame(columns=['c_tariff_plan', 'c_tariff', 'tariff_fix_id', 'commission_monthly_fix', 'rn_sum_desc'])

    tariff_fix_map_df = (
        tariff_tune_fix_details_df
        .copy()
        .assign(commission_monthly_fix=lambda x: pd.to_numeric(x['commission_monthly_fix'], errors='coerce'))
        .dropna(subset=['c_tariff_plan'])
        .groupby('c_tariff_plan', as_index=False)['commission_monthly_fix']
        .max()
        .sort_values('c_tariff_plan')
    )

print('tariff_tune_fix_details rows =', len(tariff_tune_fix_details_df))
print('tariff_fix_map rows =', len(tariff_fix_map_df))
display(tariff_tune_fix_details_df)
display(tariff_fix_map_df)

In [ ]:
# 6) Реплика логики секции 09_actual_tariff_by_agr для выбранных agr_id
sql_actual_tariff_09 = f"""
with agr_keys as (
  select distinct cast(a.abs_agr_id as string) as agr_id
  from ods_alpha.scd1_agreements a
  where cast(a.abs_agr_id as string) in ({a_ids_sql})
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
    )
),
agr_actual as (
  select
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr_actual,
    cast(a.c_agr_number as string) as contract_number_acq,
    cast(a.d_valid_from as date) as d_valid_from_actual,
    cast(a.d_valid_to as date) as d_valid_to_actual,
    row_number() over (
      partition by cast(a.abs_agr_id as string)
      order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
    ) as rn
  from ods_alpha.scd1_agreements a
  join agr_keys k
    on k.agr_id = cast(a.abs_agr_id as string)
  where cast(a.d_valid_from as date) <= cast('{month_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_end}' as date))
    and coalesce(a.ods_deleted_flg, '0') <> '1'
),
merchant_one as (
  select
    cast(m.id as string) as agr_id,
    cast(m.c_tariff_plan as string) as c_tariff_plan,
    row_number() over (
      partition by cast(m.id as string)
      order by cast(m.c_tariff_plan as string) desc
    ) as rn
  from ods.scd1_z_r2_ip_merchants m
  join agr_keys k
    on k.agr_id = cast(m.id as string)
)
select
  m.agr_id,
  aa.n_agr_actual,
  aa.contract_number_acq,
  aa.d_valid_from_actual,
  aa.d_valid_to_actual,
  m.c_tariff_plan,
  cast(tp.c_name as string) as tariff_name_actual
from merchant_one m
left join agr_actual aa
  on aa.agr_id = m.agr_id
 and aa.rn = 1
left join ods.scd1_z_r2_tariff_plan tp
  on cast(tp.id as string) = m.c_tariff_plan
where m.rn = 1
order by m.agr_id
"""

actual_tariff_09_df = run_impala(sql_actual_tariff_09, mem_limit='8g')
if actual_tariff_09_df is None:
    actual_tariff_09_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual',
        'd_valid_to_actual', 'c_tariff_plan', 'tariff_name_actual'
    ])

print('actual_tariff_09 rows =', len(actual_tariff_09_df))
display(actual_tariff_09_df)

In [ ]:
# 7) Реплика 08b (scoped by agr_id) + финальная склейка пути
sql_08b_scoped = f"""
with plan_scope as (
  select distinct cast(m.c_tariff_plan as string) as c_tariff_plan
  from ods.scd1_z_r2_ip_merchants m
  where m.c_tariff_plan is not null
    and coalesce(cast(m.ods_deleted_flg as string), '0') <> '1'
    and cast(m.id as string) in ({a_ids_sql})
)
select
  ps.c_tariff_plan,
  max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
from plan_scope ps
left join ods.scd1_z_r2_tariff_tune tt
  on cast(tt.c_tariff_plan as string) = ps.c_tariff_plan
left join ods.scd1_z_r2_tariff_fix tf
  on tt.c_tariff = tf.id
group by ps.c_tariff_plan
order by ps.c_tariff_plan
"""

fix_08b_df = run_impala(sql_08b_scoped, mem_limit='8g')
if fix_08b_df is None:
    fix_08b_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

if not fix_08b_df.empty:
    fix_08b_df['commission_monthly_fix'] = pd.to_numeric(fix_08b_df['commission_monthly_fix'], errors='coerce')

final_path_df = actual_tariff_09_df.copy()
if not final_path_df.empty:
    final_path_df = final_path_df.merge(fix_08b_df, on='c_tariff_plan', how='left')

print('fix_08b rows =', len(fix_08b_df))
display(fix_08b_df)

print('Final path agr_id -> tariff -> monthly commission:')
display(final_path_df)

In [ ]:
# 8) Короткая диагностика по стадиям для каждого agr_id
stage_rows = []
for agr in a_ids:
    stage_rows.append({
        'agr_id': agr,
        'in_agreements_history': int((agreements_history_df['agr_id'].astype(str) == agr).any()) if len(agreements_history_df) else 0,
        'is_active_sa_with_p_terms': int(((active_sa_terms_df['agr_id'].astype(str) == agr) & (pd.to_numeric(active_sa_terms_df['is_sa_active_in_month'], errors='coerce') == 1) & (pd.to_numeric(active_sa_terms_df['has_active_p_term'], errors='coerce') == 1)).any()) if len(active_sa_terms_df) else 0,
        'in_r2_merchants': int((r2_merchants_df['agr_id'].astype(str) == agr).any()) if len(r2_merchants_df) else 0,
        'in_actual_tariff_09': int((actual_tariff_09_df['agr_id'].astype(str) == agr).any()) if len(actual_tariff_09_df) else 0,
        'has_commission_monthly_fix': int(((final_path_df['agr_id'].astype(str) == agr) & pd.to_numeric(final_path_df['commission_monthly_fix'], errors='coerce').notna()).any()) if len(final_path_df) else 0,
    })

stage_df = pd.DataFrame(stage_rows)
display(stage_df)